# Gold Layer — Subscriber, Content & Ad Revenue KPIs
Produces 6 gold tables: daily subscriber metrics, content performance, churn analysis, ad revenue summary, weekly trends, and subscriber scorecards.

In [ ]:
from pyspark.sql.functions import (
    col, count, sum as spark_sum, avg, round as spark_round,
    countDistinct, when, lit, date_trunc, current_timestamp,
    min as spark_min, max as spark_max
)

df_subs   = spark.read.format('delta').table('silver_subscribers')
df_views  = spark.read.format('delta').table('silver_viewing_history')
df_ads    = spark.read.format('delta').table('silver_ad_impressions')
df_content= spark.read.format('delta').table('silver_content_catalog')

In [ ]:
# Gold 1 — Daily subscriber metrics (by plan and region)
daily_subs = (
    df_subs
    .groupBy('plan_type', 'region', 'plan_tier')
    .agg(
        count('subscriber_id').alias('total_subscribers'),
        spark_sum('is_churned').alias('churned_subscribers'),
        spark_round(spark_sum('monthly_fee'), 2).alias('monthly_recurring_revenue'),
    )
    .withColumn('active_subscribers',
        col('total_subscribers') - col('churned_subscribers')
    )
    .withColumn('churn_rate',
        spark_round(col('churned_subscribers') / col('total_subscribers') * 100, 2)
    )
    .withColumn('arpu',
        spark_round(col('monthly_recurring_revenue') / col('active_subscribers'), 2)
    )
    .withColumn('gold_timestamp', current_timestamp())
)
daily_subs.write.mode('overwrite').format('delta').saveAsTable('gold_subscriber_metrics')
print(f'Gold subscriber metrics: {spark.table("gold_subscriber_metrics").count()} rows')

In [ ]:
# Gold 2 — Content performance (views, completion, ratings)
content_perf = (
    df_views
    .join(df_content.select('content_id', 'genre', 'content_type', 'language', 'release_year'), 'content_id', 'left')
    .groupBy('content_id', 'genre', 'content_type', 'language', 'release_year')
    .agg(
        count('view_id').alias('total_views'),
        spark_round(avg('watch_duration_mins'), 2).alias('avg_watch_mins'),
        spark_round(avg('completion_pct'), 2).alias('avg_completion_pct'),
        spark_sum(when(col('engagement_band') == 'Completed', 1).otherwise(0)).alias('completions'),
        spark_round(avg(when(col('rating').isNotNull(), col('rating'))), 2).alias('avg_rating'),
        countDistinct('subscriber_id').alias('unique_viewers'),
    )
    .withColumn('completion_rate',
        spark_round(col('completions') / col('total_views') * 100, 2)
    )
    .withColumn('gold_timestamp', current_timestamp())
)
content_perf.write.mode('overwrite').format('delta').saveAsTable('gold_content_performance')
print(f'Gold content performance: {spark.table("gold_content_performance").count()} rows')

In [ ]:
# Gold 3 — Churn analysis by tenure bucket and plan
churn = (
    df_subs
    .groupBy('tenure_bucket', 'plan_type', 'region', 'age_group')
    .agg(
        count('subscriber_id').alias('total_subs'),
        spark_sum('is_churned').alias('churned'),
        spark_round(avg('tenure_days'), 0).alias('avg_tenure_days'),
        spark_round(spark_sum('monthly_fee'), 2).alias('total_mrr'),
    )
    .withColumn('churn_rate',
        spark_round(col('churned') / col('total_subs') * 100, 2)
    )
    .withColumn('at_risk_revenue',
        spark_round(col('total_mrr') * col('churn_rate') / 100, 2)
    )
    .withColumn('gold_timestamp', current_timestamp())
)
churn.write.mode('overwrite').format('delta').saveAsTable('gold_churn_analysis')
print(f'Gold churn analysis: {spark.table("gold_churn_analysis").count()} rows')

In [ ]:
# Gold 4 — Ad revenue by type and content genre
ad_rev = (
    df_ads
    .join(df_content.select('content_id', 'genre', 'content_type'), 'content_id', 'left')
    .groupBy('ad_type', 'genre', 'ctr_band')
    .agg(
        count('impression_id').alias('total_impressions_records'),
        spark_sum('impressions').alias('total_impressions'),
        spark_sum('clicks').alias('total_clicks'),
        spark_round(spark_sum('revenue_usd'), 4).alias('total_revenue'),
        spark_round(avg('cpm'), 2).alias('avg_cpm'),
        spark_round(avg('ctr'), 3).alias('avg_ctr'),
    )
    .withColumn('ecpm',
        spark_round(col('total_revenue') / col('total_impressions') * 1000, 4)
    )
    .withColumn('gold_timestamp', current_timestamp())
)
ad_rev.write.mode('overwrite').format('delta').saveAsTable('gold_ad_revenue')
print(f'Gold ad revenue: {spark.table("gold_ad_revenue").count()} rows')

In [ ]:
# Gold 5 — Weekly viewing trends
weekly = (
    df_views
    .withColumn('week', date_trunc('week', col('view_date')))
    .groupBy('week')
    .agg(
        count('view_id').alias('total_views'),
        countDistinct('subscriber_id').alias('active_viewers'),
        countDistinct('content_id').alias('content_titles_viewed'),
        spark_round(avg('completion_pct'), 2).alias('avg_completion_pct'),
        spark_round(avg('watch_duration_mins'), 2).alias('avg_watch_mins'),
    )
    .orderBy('week')
    .withColumn('gold_timestamp', current_timestamp())
)
weekly.write.mode('overwrite').format('delta').saveAsTable('gold_weekly_viewing_trends')
print(f'Gold weekly viewing trends: {spark.table("gold_weekly_viewing_trends").count()} rows')

In [ ]:
# Gold 6 — Subscriber scorecards (per plan × region summary)
scorecards = (
    df_subs
    .groupBy('plan_type', 'region')
    .agg(
        count('subscriber_id').alias('total_subscribers'),
        spark_sum('is_churned').alias('churned_count'),
        spark_round(spark_sum('monthly_fee'), 2).alias('total_mrr'),
        spark_round(avg('tenure_days'), 0).alias('avg_tenure_days'),
    )
    .withColumn('active_subscribers',
        col('total_subscribers') - col('churned_count')
    )
    .withColumn('churn_rate',
        spark_round(col('churned_count') / col('total_subscribers') * 100, 2)
    )
    .withColumn('arpu',
        spark_round(col('total_mrr') / col('active_subscribers'), 2)
    )
    .withColumn('health_band',
        when(col('churn_rate') < 5, 'Healthy')
        .when(col('churn_rate') < 12, 'Monitor')
        .when(col('churn_rate') < 20, 'At Risk')
        .otherwise('Critical')
    )
    .withColumn('gold_timestamp', current_timestamp())
)
scorecards.write.mode('overwrite').format('delta').saveAsTable('gold_subscriber_scorecards')
print(f'Gold subscriber scorecards: {spark.table("gold_subscriber_scorecards").count()} rows')

In [ ]:
print('Gold aggregation complete.')
for tbl in ['gold_subscriber_metrics', 'gold_content_performance', 'gold_churn_analysis',
            'gold_ad_revenue', 'gold_weekly_viewing_trends', 'gold_subscriber_scorecards']:
    n = spark.read.format('delta').table(tbl).count()
    print(f'  {tbl:<35} {n:>6,} rows')